In [ ]:
-- #country_dim.sql

{{ config(materialized='table') }}

SELECT 
    c1.country_name,
    c1.alpha_3_code AS country_code,
    c2.region,
    c2.continent,
    c2.area_km2,
    c3.income_group
FROM {{ref('country_codes')}} AS c1
LEFT JOIN {{ref('country_geography')}} as c2
ON c1.alpha_3_code = c2.country_code
LEFT JOIN {{ref('country_income_groups')}} as c3
ON c1.alpha_3_code = c3.country_code

In [ ]:
-- country_population_facts.sql

{{ config(materialized='table') }}

WITH full_data AS (
    SELECT 
        w1.country_name, 
        w1.country_code, 
        w1.2020 AS pop_2020,
        w1.2021 AS pop_2021,
        w1.2022 AS pop_2022,
        w1.2023 AS pop_2023,
        w1.2024 AS pop_2024,
        c1.area_km2
    FROM {{ref('world_population')}} as w1
    INNER JOIN {{ref('country_geography')}} as c1
    ON w1.country_code = c1.country_code
    ORDER BY country_name)

    ,population_2020 AS (
        SELECT 
            country_name, 
            country_code, 
            2020 AS year, 
            full_data.pop_2020 AS population,
            area_km2,
            ROUND(full_data.pop_2020/area_km2, 1) AS density_km2
        FROM full_data)

    ,population_2021 AS (
        SELECT 
            country_name, 
            country_code, 
            2021 AS year, 
            full_data.pop_2021 AS population,
            area_km2,
            ROUND(full_data.pop_2021/area_km2, 1) AS density_km2
        FROM full_data)

    ,population_2022 AS (
        SELECT 
            country_name, 
            country_code, 
            2022 AS year, 
            full_data.pop_2022 AS population,
            area_km2,
            ROUND(full_data.pop_2022/area_km2, 1) AS density_km2
        FROM full_data)

    ,population_2023 AS (
        SELECT 
            country_name, 
            country_code, 
            2023 AS year, 
            full_data.pop_2023 AS population,
            area_km2,
            ROUND(full_data.pop_2023/area_km2, 1) AS density_km2
        FROM full_data)

    ,population_2024 AS (
        SELECT 
            country_name, 
            country_code, 
            2024 AS year, 
            full_data.pop_2024 AS population,
            area_km2,
            ROUND(full_data.pop_2024/area_km2, 1) AS density_km2
        FROM full_data)
 
SELECT 
    country_name,
    country_code, 
    year, 
    population,
    area_km2,
    density_km2
FROM population_2020
UNION ALL 
SELECT * FROM population_2021 
UNION ALL 
SELECT * FROM population_2022 
UNION ALL 
SELECT * FROM population_2023 
UNION ALL 
SELECT * FROM population_2024
ORDER BY country_name , year

In [ ]:
-- covid19_full_data_metrics.sql

{{ config(
    materialized='table',
    partition_by={
        "field": "date",             
        "data_type": "date"           
    },
    cluster_by=['country']
) }}


WITH CT1 AS (
  SELECT
    c1.country,
    c2.country_code,
    c1.region,
    c2.continent,
    c3.continent_top_index,
    c2.income_group, 
    EXTRACT(YEAR FROM date) AS year,
    FORMAT_DATE('%Y-%m', date) AS year_month,
    c1.date,
    c1.new_cases,
    c1.total_cases,
    c1.new_deaths,
    c1.total_deaths
  FROM {{ref('covid19_cases_deaths_all_years_region')}} as c1
  LEFT JOIN {{ref('country_dim')}} as c2
  ON c1.country = c2.country_name
  LEFT JOIN {{ref('continent_index_table')}} as c3
  ON c2.continent = c3.continent
)

--Adding weekly/beweekly/monthly new_cases and new_deaths
 ,CTE2 AS (
  SELECT
    c1.country,
    c1.country_code,
    c1.region,
    c1.continent,
    c1.continent_top_index,
    c1.income_group,
    c1.date,
    EXTRACT(YEAR FROM date) AS year,
    FORMAT_DATE('%Y-%m', date) AS year_month,
    c1.new_cases,
    c1.total_cases,
    c1.new_deaths,
    c1.total_deaths,
    c2.population,
    c2.area_km2,
    c2.density_km2,
    SUM(c1.new_cases) OVER (PARTITION BY c1.country ORDER BY c1.date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS weekly_new_cases,
    SUM(c1.new_deaths) OVER (PARTITION BY c1.country ORDER BY c1.date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS weekly_new_deaths,
    SUM(c1.new_cases) OVER (PARTITION BY c1.country ORDER BY date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW) AS biweekly_new_cases,
    SUM(c1.new_deaths) OVER (PARTITION BY c1.country ORDER BY date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW) AS biweekly_new_deaths,
    SUM(c1.new_cases) OVER (PARTITION BY c1.country ORDER BY c1.date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS monthly_new_cases,
    SUM(c1.new_deaths) OVER (PARTITION BY c1.country ORDER BY c1.date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS monthly_new_deaths,
    AVG(c1.new_cases) OVER (PARTITION BY c1.country ORDER BY c1.date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS weekly_avg_new_cases,
    AVG(c1.new_deaths) OVER (PARTITION BY c1.country ORDER BY c1.date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS weekly_avg_new_deaths,
    AVG(c1.new_cases) OVER (PARTITION BY c1.country ORDER BY date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW) AS biweekly_avg_new_cases,
    AVG(c1.new_deaths) OVER (PARTITION BY c1.country ORDER BY date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW) AS biweekly_avg_new_deaths,
    AVG(c1.new_cases) OVER (PARTITION BY c1.country ORDER BY c1.date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS monthly_avg_new_cases,
    AVG(c1.new_deaths) OVER (PARTITION BY c1.country ORDER BY c1.date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS monthly_avg_new_deaths
  FROM CT1 AS c1
  LEFT JOIN {{ref('country_population_facts')}} AS c2
    ON c1.country_code = c2.country_code
  WHERE c2.year = EXTRACT(YEAR FROM date)
)

--Calculating covid metrics
SELECT 
  country,
  country_code,
  region,
  continent,
  continent_top_index,
  income_group,
  year,
  year_month,
  date,
  new_cases AS daily_new_cases,
  total_cases,
  new_deaths AS daily_new_deaths,
  total_deaths,
  population,
  area_km2,
  density_km2,
  -- Metrics
  weekly_new_cases,
  weekly_new_deaths,
  biweekly_new_cases,
  biweekly_new_deaths,
  monthly_new_cases,
  monthly_new_deaths,
  weekly_avg_new_cases,
  weekly_avg_new_deaths,
  biweekly_avg_new_cases,
  biweekly_avg_new_deaths,
  monthly_avg_new_cases,
  monthly_avg_new_deaths,
  ROUND(SAFE_DIVIDE(new_cases, population), 6) * 1000000 AS daily_new_cases_per_million,
  ROUND(SAFE_DIVIDE(new_deaths, population), 6) * 1000000 AS daily_new_deaths_per_million,
  ROUND(SAFE_DIVIDE(weekly_new_cases, population), 6) * 1000000 AS weekly_new_cases_per_million,
  ROUND(SAFE_DIVIDE(weekly_new_deaths, population), 6) * 1000000 AS weekly_new_deaths_per_million,
  ROUND(SAFE_DIVIDE(biweekly_new_cases, population), 6) * 1000000 AS biweekly_new_cases_per_million,
  ROUND(SAFE_DIVIDE(biweekly_new_deaths, population), 6) * 1000000 AS biweekly_new_deaths_per_million,
  ROUND(SAFE_DIVIDE(monthly_new_cases, population), 6) * 1000000 AS monthly_new_cases_per_million,
  ROUND(SAFE_DIVIDE(monthly_new_deaths, population), 6) * 1000000 AS monthly_new_deaths_per_million,
  ROUND(SAFE_DIVIDE(total_cases, population), 6) * 1000000 AS total_cases_per_million,
  ROUND(SAFE_DIVIDE(total_deaths, population), 6) * 1000000 AS total_deaths_per_million,
  ROUND(SAFE_DIVIDE(total_cases, population), 4) * 100 AS total_infection_rate_pct,
  ROUND(SAFE_DIVIDE(total_deaths, total_cases), 4) * 100 AS total_mortality_rate_pct,
  ROUND(SAFE_DIVIDE(weekly_new_cases, population), 4) * 100 AS weekly_infection_rate_pct,
  ROUND(SAFE_DIVIDE(weekly_new_deaths, weekly_new_cases), 4) * 100 AS weekly_mortality_rate_pct,
  ROUND(SAFE_DIVIDE(biweekly_new_cases, population), 4) * 100 AS biweekly_infection_rate_pct,
  ROUND(SAFE_DIVIDE(biweekly_new_deaths, biweekly_new_cases), 4) * 100 AS biweekly_mortality_rate_pct,
  ROUND(SAFE_DIVIDE(monthly_new_cases, population), 4) * 100 AS monthly_infection_rate_pct,
  ROUND(SAFE_DIVIDE(monthly_new_deaths, monthly_new_cases), 4) * 100 AS monthly_mortality_rate_pct
FROM CTE2

In [ ]:
-- covid19_cases_deaths_all_years_region.sql


{{ config(
    materialized='table',
    partition_by={
        "field": "date",             
        "data_type": "date"           
    },
    cluster_by=['country']
) }}

WITH cases_deaths_2020 AS(
  SELECT *
  FROM `covid19-dbt-analytics-2.dev_covid19_raw.cases_deaths_*`
  WHERE _TABLE_SUFFIX IN (
    '2020_africa', 
    '2020_americas', 
    '2020_eastern_mediterranean', 
    '2020_europe', 
    '2020_south_east_asia', 
    '2020_western_pacific')
  )
  ,cases_deaths_2021 AS(
  SELECT *
  FROM `covid19-dbt-analytics-2.dev_covid19_raw.cases_deaths_*`
  WHERE _TABLE_SUFFIX IN (
    '2021_africa', 
    '2021_americas', 
    '2021_eastern_mediterranean', 
    '2021_europe', 
    '2021_south_east_asia', 
    '2021_western_pacific')
  )
  , cases_deaths_2022 AS(
  SELECT *
  FROM `covid19-dbt-analytics-2.dev_covid19_raw.cases_deaths_*`
  WHERE _TABLE_SUFFIX IN (
    '2022_africa', 
    '2022_americas', 
    '2022_eastern_mediterranean', 
    '2022_europe', 
    '2022_south_east_asia', 
    '2022_western_pacific')
  )
  , cases_deaths_2023 AS(
  SELECT *
  FROM `covid19-dbt-analytics-2.dev_covid19_raw.cases_deaths_*`
  WHERE _TABLE_SUFFIX IN (
    '2023_africa', 
    '2023_americas', 
    '2023_eastern_mediterranean', 
    '2023_europe', 
    '2023_south_east_asia', 
    '2023_western_pacific')
  )
  , cases_deaths_2024 AS(
  SELECT *
  FROM `covid19-dbt-analytics-2.dev_covid19_raw.cases_deaths_*`
  WHERE _TABLE_SUFFIX IN (
    '2024_africa', 
    '2024_americas', 
    '2024_eastern_mediterranean', 
    '2024_europe', 
    '2024_south_east_asia', 
    '2024_western_pacific')
  )
  , cases_deaths_2025 AS(
  SELECT *
  FROM `covid19-dbt-analytics-2.dev_covid19_raw.cases_deaths_*`
  WHERE _TABLE_SUFFIX IN (
    '2025_africa', 
    '2025_americas', 
    '2025_eastern_mediterranean', 
    '2025_europe', 
    '2025_south_east_asia', 
    '2025_western_pacific')
  )

SELECT * FROM cases_deaths_2020
UNION ALL
SELECT * FROM cases_deaths_2021
UNION ALL
SELECT * FROM cases_deaths_2022
UNION ALL
SELECT * FROM cases_deaths_2023
UNION ALL
SELECT * FROM cases_deaths_2024
UNION ALL
SELECT * FROM cases_deaths_2025